In [1]:
from torch_geometric.datasets import OGB_MAG

dataset = OGB_MAG(root="/tmp/ogb_mag", preprocess="metapath2vec")

/Users/tungvuduc/opt/anaconda3/envs/dlbio_arm64/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from torch_geometric.nn import MessagePassing
import torch.nn as nn
import torch
from torch_geometric.utils import add_self_loops, degree

class GCNConv(MessagePassing):
    def __init__(self, in_features, out_features):
        super().__init__(aggr="add")

        self.message_mlp = nn.Linear(in_features, out_features, bias=False)
        self.update_mlp = nn.Linear(out_features*2, out_features)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor):
        edge_index, _ = add_self_loops(edge_index)
        src, target = edge_index
        
        deg = degree(target, num_nodes=x.size(0))
        deg = deg.pow(-0.5)
        deg[deg == float("inf")] = 0
        norm = deg[target] * deg[src]

        x = self.message_mlp(x)

        return self.propagate(edge_index=edge_index, x=x, norm=norm)
    
    def message(self, x_j: torch.Tensor, norm: torch.Tensor):
        return  x_j * norm.view(-1, 1)

    def update(self, aggr_out: torch.Tensor, x: torch.Tensor):
        return self.update_mlp(torch.cat([x, aggr_out], dim=1))
    
gcn_conv = GCNConv(10, 20)

In [ ]:

class EdgeConv(MessagePassing):
    def __init__(self, in_features: int, out_features:int):
        super().__init__(aggr="max")

        self.mlp = nn.Sequential(
            nn.Linear(in_features*2, out_features),
            nn.ReLU(),
            nn.Linear(out_features, out_features)
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor):
        return self.propagate(edge_index=edge_index, x=x)
    
    def message(self, x_j:torch.Tensor, x_i: torch.Tensor):
        return self.mlp(torch.cat([x_i, x_j - x_i], dim=1))

HeteroData(
  paper={
    x=[736389, 128],
    year=[736389],
    y=[736389],
    train_mask=[736389],
    val_mask=[736389],
    test_mask=[736389],
  },
  author={ x=[1134649, 128] },
  institution={ x=[8740, 128] },
  field_of_study={ x=[59965, 128] },
  (author, affiliated_with, institution)={ edge_index=[2, 1043998] },
  (author, writes, paper)={ edge_index=[2, 7145660] },
  (paper, cites, paper)={ edge_index=[2, 5416271] },
  (paper, has_topic, field_of_study)={ edge_index=[2, 7505078] }
)